# Notebook 01: ETL e Modelagem Dimensional (Star Schema)

**Projeto:** Predição de Avaliações Negativas no Ecossistema Olist<br>
**Autor:** Rafael de Menezes Ehlers<br>
**Fase:** 2 (Implementação)<br>
**Curso:** Curso Superior de Tecnologia em Banco de Dados

## Objetivo

Este notebook implementa as fases CRISP-DM de **Entendimento dos Dados** e **Preparação dos Dados** sobre o dataset público da Olist. O resultado é um Data Warehouse local em SQLite estruturado em Star Schema (1 tabela fato e 5 dimensões), o qual servirá de base para para o Notebook 02 (Dashboards de BI) e o Notebook 03 (Modelo Preditivo).

## Pré-requisitos

1. Possuir uma conta Google com o Google Drive habilitado.
2. Criar as pastas `PUCRS/projetobi-olist/data/` no Google Drive.
3. Baixar o dataset Olist do Kaggle em https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce.
4. Extrair o arquivo zip do dataset.
5. Copiar os 9 arquivos CSV do dataset e colá-los dentro da pasta `PUCRS/projetobi-olist/data/`.
   

## Saída

O arquivo `olist_dw.sqlite` será criado dentro da pasta `/content/drive/MyDrive/PUCRS/projetobi-olist/`



## 1. Setup do ambiente

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine

# Caminhos no Google Drive
BASE_PATH = '/content/drive/MyDrive/PUCRS/projetobi-olist'
DATA_PATH = os.path.join(BASE_PATH, 'data')
DW_PATH = os.path.join(BASE_PATH, 'olist_dw.sqlite')

# Cria as pastas se nao existirem
os.makedirs(DATA_PATH, exist_ok=True)

print(f'BASE_PATH: {BASE_PATH}')
print(f'DATA_PATH: {DATA_PATH}')
print(f'DW_PATH:   {DW_PATH}')

BASE_PATH: /content/drive/MyDrive/PUCRS/projetobi-olist
DATA_PATH: /content/drive/MyDrive/PUCRS/projetobi-olist/data
DW_PATH:   /content/drive/MyDrive/PUCRS/projetobi-olist/olist_dw.sqlite


## 2. Coleta dos dados (CRISP-DM: Entendimento dos Dados)

Carrega os 9 arquivos CSV do dataset Olist a partir do Google Drive. Cada arquivo se torna um DataFrame Pandas separado.

In [4]:
df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
df_items = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))
df_reviews = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'))
df_customers = pd.read_csv(os.path.join(DATA_PATH, 'olist_customers_dataset.csv'))
df_products = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
df_sellers = pd.read_csv(os.path.join(DATA_PATH, 'olist_sellers_dataset.csv'))
df_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
df_geolocation = pd.read_csv(os.path.join(DATA_PATH, 'olist_geolocation_dataset.csv'))
df_category_translation = pd.read_csv(os.path.join(DATA_PATH, 'product_category_name_translation.csv'))

print('CSVs carregados com sucesso.')

CSVs carregados com sucesso.


In [5]:
# Visao geral dos volumes
datasets = {
    'orders': df_orders,
    'order_items': df_items,
    'reviews': df_reviews,
    'customers': df_customers,
    'products': df_products,
    'sellers': df_sellers,
    'payments': df_payments,
    'geolocation': df_geolocation,
    'category_translation': df_category_translation
}

for name, df in datasets.items():
    print(f'{name:25s}: {df.shape[0]:>10,} linhas, {df.shape[1]:>3} colunas')

orders                   :     99,441 linhas,   8 colunas
order_items              :    112,650 linhas,   7 colunas
reviews                  :     99,224 linhas,   7 colunas
customers                :     99,441 linhas,   5 colunas
products                 :     32,951 linhas,   9 colunas
sellers                  :      3,095 linhas,   4 colunas
payments                 :    103,886 linhas,   5 colunas
geolocation              :  1,000,163 linhas,   5 colunas
category_translation     :         71 linhas,   2 colunas


## 3. Limpeza e padronização (CRISP-DM: Preparação dos Dados)

Conversão de tipos de data, filtro de pedidos efetivamente entregues e deduplicação de reviews.

In [6]:
# Converter colunas de data para datetime
date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols_orders:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'], errors='coerce')
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'], errors='coerce')

print('Datas convertidas.')
print(f'Pedidos com order_delivered_customer_date nula: {df_orders["order_delivered_customer_date"].isna().sum():,}')

Datas convertidas.
Pedidos com order_delivered_customer_date nula: 2,965


In [7]:
# Manter apenas pedidos efetivamente entregues (base do calculo de atraso e satisfacao)
df_orders_entregues = df_orders[
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_customer_date'].notna()) &
    (df_orders['order_estimated_delivery_date'].notna())
].copy()

print(f'Pedidos totais:                       {len(df_orders):>8,}')
print(f'Pedidos entregues com datas validas:  {len(df_orders_entregues):>8,}')
print(f'Reducao:                              {100 * (1 - len(df_orders_entregues) / len(df_orders)):>7.2f}%')

Pedidos totais:                         99,441
Pedidos entregues com datas validas:    96,470
Reducao:                                 2.99%


In [8]:
# Alguns pedidos tem mais de uma review; manter a mais recente
df_reviews_dedup = (
    df_reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates('order_id', keep='first')
    .copy()
)
print(f'Reviews originais:         {len(df_reviews):,}')
print(f'Reviews unicas por pedido: {len(df_reviews_dedup):,}')

Reviews originais:         99,224
Reviews unicas por pedido: 98,673


## 4. Engenharia de features

Cálculo das variáveis derivadas que alimentam as 3 hipóteses da Fase 1:

- `delta_entrega_dias` e `atraso` → base da **Hipótese 1**
- `razao_frete_preco` e `faixa_ticket` → base da **Hipótese 2**
- `num_sellers_unicos`, `multi_vendedor`, `tempo_consolidacao_dias` → base da **Hipótese 3**

A variável-alvo do modelo preditivo, `avaliacao_negativa`, é derivada como `review_score <= 3`.

In [9]:
# Agregar order_items para o nivel de pedido
agg_items = df_items.groupby('order_id').agg(
    valor_total_pedido=('price', 'sum'),
    valor_total_frete=('freight_value', 'sum'),
    num_itens=('order_item_id', 'count'),
    num_sellers_unicos=('seller_id', 'nunique'),
    seller_principal=('seller_id', 'first'),
    product_principal=('product_id', 'first')
).reset_index()

# Features derivadas a partir da agregacao
agg_items['multi_vendedor'] = (agg_items['num_sellers_unicos'] > 1).astype(int)
agg_items['razao_frete_preco'] = agg_items['valor_total_frete'] / agg_items['valor_total_pedido']

print(f'Agregacao de itens: {len(agg_items):,} pedidos')
agg_items.head()

Agregacao de itens: 98,666 pedidos


,order_id,valor_total_pedido,valor_total_frete,num_itens,num_sellers_unicos,seller_principal,product_principal,multi_vendedor,razao_frete_preco
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,0,0.225637
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,0,0.083076
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,0,0.089799
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,0,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,0,0.090745


In [10]:
# Juntar pedidos entregues + itens agregados + review
fato = df_orders_entregues.merge(agg_items, on='order_id', how='inner')
fato = fato.merge(df_reviews_dedup[['order_id', 'review_score']], on='order_id', how='left')

# Manter apenas pedidos com review (necessario para o modelo preditivo)
fato = fato[fato['review_score'].notna()].copy()
print(f'Pedidos com review: {len(fato):,}')

Pedidos com review: 95,824


In [11]:
# Delta de entrega: dias entre entrega real e prazo estimado (positivo = atrasou)
fato['delta_entrega_dias'] = (
    fato['order_delivered_customer_date'] - fato['order_estimated_delivery_date']
).dt.days

fato['atraso'] = (fato['delta_entrega_dias'] > 0).astype(int)

# Tempo total de entrega (compra ate recebimento)
fato['tempo_total_entrega_dias'] = (
    fato['order_delivered_customer_date'] - fato['order_purchase_timestamp']
).dt.days

# Tempo de consolidacao logistica (aprovacao ate envio ao carrier)
fato['tempo_consolidacao_dias'] = (
    fato['order_delivered_carrier_date'] - fato['order_approved_at']
).dt.days

print(fato[['delta_entrega_dias', 'atraso', 'tempo_total_entrega_dias', 'tempo_consolidacao_dias']].describe().round(2))

       delta_entrega_dias    atraso  tempo_total_entrega_dias  \
count            95824.00  95824.00                  95824.00   
mean               -11.91      0.07                     12.05   
std                 10.11      0.25                      9.47   
min               -147.00      0.00                      0.00   
25%                -17.00      0.00                      6.00   
50%                -12.00      0.00                     10.00   
75%                 -7.00      0.00                     15.00   
max                188.00      1.00                    208.00   

       tempo_consolidacao_dias  
count                 95809.00  
mean                      2.29  
std                       3.53  
min                    -172.00  
25%                       0.00  
50%                       1.00  
75%                       3.00  
max                     125.00  


In [12]:
# Faixa de ticket (conforme definicao da Fase 1)
def faixa_ticket(valor):
    if valor < 50:
        return 'baixo'
    elif valor < 200:
        return 'medio'
    return 'alto'

fato['faixa_ticket'] = fato['valor_total_pedido'].apply(faixa_ticket)

# Variavel alvo: avaliacao negativa (notas 1, 2 ou 3)
fato['avaliacao_negativa'] = (fato['review_score'] <= 3).astype(int)

print('Distribuicao da variavel alvo (% por classe):')
print((fato['avaliacao_negativa'].value_counts(normalize=True).round(4) * 100), '\n')

print('Distribuicao por faixa de ticket (% por faixa):')
print((fato['faixa_ticket'].value_counts(normalize=True).round(4) * 100))

Distribuicao da variavel alvo (% por classe):
avaliacao_negativa
0    78.93
1    21.07
Name: proportion, dtype: float64 

Distribuicao por faixa de ticket (% por faixa):
faixa_ticket
medio    54.98
baixo    29.87
alto     15.15
Name: proportion, dtype: float64


In [13]:
# Chave de tempo para join com dim_tempo (formato YYYYMMDD)
fato['data_compra_key'] = fato['order_purchase_timestamp'].dt.strftime('%Y%m%d').astype(int)

## 5. Modelagem dimensional (Star Schema)

Construção da tabela fato `fato_pedidos` e das 5 dimensões: `dim_cliente`, `dim_produto`, `dim_vendedor`, `dim_tempo`, `dim_geolocalizacao`. Cada dimensão mantém apenas os registros referenciados pela fato (filtro de subset relevante).

In [14]:
dim_cliente = df_customers.copy()
dim_cliente.columns = ['customer_id', 'customer_unique_id', 'cep_cliente', 'cidade_cliente', 'estado_cliente']
dim_cliente = dim_cliente[dim_cliente['customer_id'].isin(fato['customer_id'])].drop_duplicates('customer_id')
print(f'dim_cliente: {len(dim_cliente):,} linhas')
dim_cliente.head()

dim_cliente: 95,824 linhas


,customer_id,customer_unique_id,cep_cliente,cidade_cliente,estado_cliente
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [15]:
dim_produto = df_products.merge(df_category_translation, on='product_category_name', how='left')
dim_produto = dim_produto[['product_id', 'product_category_name', 'product_category_name_english', 'product_weight_g']]
dim_produto.columns = ['product_id', 'categoria', 'categoria_en', 'peso_g']
dim_produto = dim_produto[dim_produto['product_id'].isin(fato['product_principal'])].drop_duplicates('product_id')
print(f'dim_produto: {len(dim_produto):,} linhas')
dim_produto.head()

dim_produto: 31,012 linhas


,product_id,categoria,categoria_en,peso_g
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,225.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,1000.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,154.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby,371.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,625.0


In [16]:
dim_vendedor = df_sellers.copy()
dim_vendedor.columns = ['seller_id', 'cep_vendedor', 'cidade_vendedor', 'estado_vendedor']
dim_vendedor = dim_vendedor[dim_vendedor['seller_id'].isin(fato['seller_principal'])].drop_duplicates('seller_id')
print(f'dim_vendedor: {len(dim_vendedor):,} linhas')
dim_vendedor.head()

dim_vendedor: 2,956 linhas


,seller_id,cep_vendedor,cidade_vendedor,estado_vendedor
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [17]:
datas_unicas = pd.Series(fato['order_purchase_timestamp'].dt.normalize().unique())
dim_tempo = pd.DataFrame({'data': datas_unicas})
dim_tempo['data_key'] = dim_tempo['data'].dt.strftime('%Y%m%d').astype(int)
dim_tempo['ano'] = dim_tempo['data'].dt.year
dim_tempo['mes'] = dim_tempo['data'].dt.month
dim_tempo['dia'] = dim_tempo['data'].dt.day
dim_tempo['dia_semana'] = dim_tempo['data'].dt.day_name()
dim_tempo['trimestre'] = dim_tempo['data'].dt.quarter
dim_tempo = dim_tempo[['data_key', 'data', 'ano', 'mes', 'dia', 'dia_semana', 'trimestre']].sort_values('data_key')
print(f'dim_tempo: {len(dim_tempo):,} linhas')
dim_tempo.head()

dim_tempo: 612 linhas


,data_key,data,ano,mes,dia,dia_semana,trimestre
607,20160915,2016-09-15,2016,9,15,Thursday,3
599,20161003,2016-10-03,2016,10,3,Monday,4
509,20161004,2016-10-04,2016,10,4,Tuesday,4
256,20161005,2016-10-05,2016,10,5,Wednesday,4
592,20161006,2016-10-06,2016,10,6,Thursday,4


In [18]:
# Agregacao de lat/lng por CEP (um CEP pode ter varias coordenadas no dataset bruto)
dim_geo = df_geolocation.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean'),
    cidade=('geolocation_city', 'first'),
    estado=('geolocation_state', 'first')
).reset_index()
dim_geo.columns = ['cep', 'lat', 'lng', 'cidade', 'estado']
print(f'dim_geolocalizacao: {len(dim_geo):,} linhas')
dim_geo.head()

dim_geolocalizacao: 19,015 linhas


,cep,lat,lng,cidade,estado
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


In [19]:
# Selecao e renomeacao final das colunas da fato
fato_pedidos = fato[[
    'order_id', 'customer_id', 'seller_principal', 'product_principal',
    'data_compra_key', 'review_score', 'avaliacao_negativa',
    'delta_entrega_dias', 'atraso',
    'tempo_total_entrega_dias', 'tempo_consolidacao_dias',
    'valor_total_pedido', 'valor_total_frete', 'razao_frete_preco',
    'num_itens', 'num_sellers_unicos', 'multi_vendedor', 'faixa_ticket'
]].copy()

fato_pedidos.columns = [
    'order_id', 'cliente_key', 'vendedor_key', 'produto_key',
    'tempo_key', 'review_score', 'avaliacao_negativa',
    'delta_entrega_dias', 'atraso',
    'tempo_total_entrega_dias', 'tempo_consolidacao_dias',
    'valor_total_pedido', 'valor_total_frete', 'razao_frete_preco',
    'num_itens', 'num_sellers_unicos', 'multi_vendedor', 'faixa_ticket'
]

print(f'fato_pedidos: {len(fato_pedidos):,} linhas, {len(fato_pedidos.columns)} colunas')
fato_pedidos.head()

fato_pedidos: 95,824 linhas, 18 colunas


,order_id,cliente_key,vendedor_key,produto_key,tempo_key,review_score,avaliacao_negativa,delta_entrega_dias,atraso,tempo_total_entrega_dias,tempo_consolidacao_dias,valor_total_pedido,valor_total_frete,razao_frete_preco,num_itens,num_sellers_unicos,multi_vendedor,faixa_ticket
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,20171002,4.0,0,-8,0,8,2.0,29.99,8.72,0.290764,1,1,0,baixo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,20180724,4.0,0,-6,0,13,0.0,118.70,22.76,0.191744,1,1,0,medio
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,20180808,5.0,0,-18,0,9,0.0,159.90,19.22,0.120200,1,1,0,medio
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,66922902710d126a0e7d26b0e3805106,d0b61bfb1de832b15ba9d266ca96e5b0,20171118,5.0,0,-13,0,13,3.0,45.00,27.20,0.604444,1,1,0,baixo
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2c9e548be18521d1c43cde1c582c6de8,65266b2da20d04dbe00c5c2d3bb7859e,20180213,5.0,0,-10,0,2,0.0,19.90,8.72,0.438191,1,1,0,baixo


## 6. Persistência no Data Warehouse SQLite

Grava todas as tabelas (fato + dimensões) no arquivo SQLite que será consumido pelos Notebooks 02 e 03.

In [20]:
engine = create_engine(f'sqlite:///{DW_PATH}')

fato_pedidos.to_sql('fato_pedidos', engine, if_exists='replace', index=False)
dim_cliente.to_sql('dim_cliente', engine, if_exists='replace', index=False)
dim_produto.to_sql('dim_produto', engine, if_exists='replace', index=False)
dim_vendedor.to_sql('dim_vendedor', engine, if_exists='replace', index=False)
dim_tempo.to_sql('dim_tempo', engine, if_exists='replace', index=False)
dim_geo.to_sql('dim_geolocalizacao', engine, if_exists='replace', index=False)

print(f'Star Schema persistido em: {DW_PATH}')
print(f'Tamanho do arquivo: {os.path.getsize(DW_PATH) / (1024**2):.2f} MB')

Star Schema persistido em: /content/drive/MyDrive/PUCRS/projetobi-olist/olist_dw.sqlite
Tamanho do arquivo: 29.42 MB


## 7. Validação: KPIs principais e teste rápido das hipóteses

Roda queries SQL diretamente sobre o Data Warehouse para validar a integridade dos dados e fazer um primeiro teste exploratório das 3 hipóteses da Fase 1. Os resultados completos virão nos Notebooks 02 (dashboards) e 03 (modelo preditivo).

In [21]:
# KPIs principais (correspondem a Secao 1 do relatorio)
query_kpis = '''
SELECT
    COUNT(*) AS total_pedidos,
    ROUND(SUM(valor_total_pedido), 2) AS faturamento_total,
    ROUND(AVG(valor_total_pedido), 2) AS ticket_medio,
    ROUND(AVG(tempo_total_entrega_dias), 2) AS prazo_medio_dias,
    ROUND(100.0 * SUM(atraso) / COUNT(*), 2) AS taxa_atraso_pct,
    ROUND(100.0 * SUM(avaliacao_negativa) / COUNT(*), 2) AS taxa_avaliacao_negativa_pct,
    ROUND(100.0 * SUM(multi_vendedor) / COUNT(*), 2) AS taxa_multi_vendedor_pct
FROM fato_pedidos
'''
pd.read_sql(query_kpis, engine)

,total_pedidos,faturamento_total,ticket_medio,prazo_medio_dias,taxa_atraso_pct,taxa_avaliacao_negativa_pct,taxa_multi_vendedor_pct
0,95824,13108744.45,136.8,12.05,6.66,21.07,1.32


In [25]:
# H1: atraso na entrega aumenta a probabilidade de avaliacao negativa?
h1 = pd.read_sql('''
SELECT
    CASE WHEN atraso = 1 THEN 'Atrasado' ELSE 'No prazo' END AS situacao,
    COUNT(*) AS n_pedidos,
    ROUND(100.0 * AVG(avaliacao_negativa), 2) AS taxa_neg_pct
FROM fato_pedidos
GROUP BY atraso
''', engine)
print('H1 - Atraso versus avaliação negativa:')
print(h1)

H1 - Atraso versus avaliação negativa:
   situacao  n_pedidos  taxa_neg_pct
0  No prazo      89443         17.34
1  Atrasado       6381         73.28


In [26]:
# H2: em ticket baixo, o frete pesa mais? Olhamos a razao frete/preco por faixa
h2 = pd.read_sql('''
SELECT
    faixa_ticket,
    COUNT(*) AS n_pedidos,
    ROUND(AVG(razao_frete_preco), 3) AS razao_frete_preco_media,
    ROUND(100.0 * AVG(avaliacao_negativa), 2) AS taxa_neg_pct
FROM fato_pedidos
GROUP BY faixa_ticket
ORDER BY CASE faixa_ticket WHEN 'baixo' THEN 1 WHEN 'medio' THEN 2 ELSE 3 END
''', engine)
print('H2 - Faixa de ticket versus frete/preço e avaliação negativa:')
print(h2)

H2 - Faixa de ticket versus frete/preço e avaliação negativa:
  faixa_ticket  n_pedidos  razao_frete_preco_media  taxa_neg_pct
0        baixo      28622                    0.570         19.65
1        medio      52681                    0.221         20.98
2         alto      14521                    0.109         24.16


In [27]:
# H3: pedidos multi-vendedor tem mais avaliacoes negativas?
h3 = pd.read_sql('''
SELECT
    CASE WHEN multi_vendedor = 1 THEN 'Múltiplos vendedores' ELSE 'Único vendedor' END AS tipo,
    COUNT(*) AS n_pedidos,
    ROUND(AVG(tempo_consolidacao_dias), 2) AS tempo_consolidacao_medio,
    ROUND(100.0 * AVG(avaliacao_negativa), 2) AS taxa_neg_pct
FROM fato_pedidos
GROUP BY multi_vendedor
''', engine)
print('H3 - Múltiplos vendedores versus tempo de consolidação e avaliação negativa:')
print(h3)

H3 - Múltiplos vendedores versus tempo de consolidação e avaliação negativa:
                   tipo  n_pedidos  tempo_consolidacao_medio  taxa_neg_pct
0        Único vendedor      94563                      2.30         20.54
1  Múltiplos vendedores       1261                      1.38         60.51
